# **|| Sentiment Analysis — Traditional ML vs BERT ||**

A sentiment analysis project on the IMDB Movie Reviews dataset comparing classical machine learning approaches against a modern transformer model.

**Models compared:**
- Logistic Regression (TF-IDF)
- Naive Bayes (TF-IDF)
- Support Vector Machine (TF-IDF)
- BERT (DistilBERT fine-tuned)

**Dataset:** IMDB Movie Reviews — 50,000 reviews, binary sentiment (positive/negative)

**Tech used:** `scikit-learn` · `transformers` · `torch` · `gradio`


## **1** - ***Installation***

In [ ]:
%%capture
!pip install transformers datasets torch scikit-learn gradio matplotlib seaborn

## **2** - ***Imports and Settings***

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

os.environ["PYTHONIOENCODING"] = "utf-8"

# --- Settings ---
SAMPLE_SIZE = 5000       # Number of reviews to use — increase for better accuracy but slower training
MAX_FEATURES = 10000     # Max number of words for TF-IDF vectorizer
RANDOM_STATE = 42        # For reproducibility
TEST_SIZE = 0.2          # 80% train, 20% test

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")
print("Imports done!")

## **3** - ***Load and Explore the Dataset***

In [ ]:
print("Loading IMDB dataset...")
# Loads directly from HuggingFace. No API key needed.
dataset = load_dataset("imdb")

# Convert to pandas for easier handling
train_df = pd.DataFrame(dataset["train"])
test_df  = pd.DataFrame(dataset["test"])

print(f"\nTrain size: {len(train_df)}")
print(f"Test size: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
print(f"\nSample review:")
print(train_df["text"].iloc[0][:300], "...")
print(f"Label: {'Positive' if train_df['label'].iloc[0] == 1 else 'Negative'}")

In [ ]:
# Check class distribution
print("Class distribution in training set:")
print(train_df["label"].value_counts())
print("\n0 = Negative, 1 = Positive")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
train_df["label"].map({0: "Negative", 1: "Positive"}).value_counts().plot(
    kind="bar", ax=axes[0], color=["#e74c3c", "#2ecc71"], edgecolor="black"
)
axes[0].set_title("Sentiment Distribution")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
axes[0].tick_params(rotation=0)

# Review length distribution
train_df["review_length"] = train_df["text"].apply(lambda x: len(x.split()))
axes[1].hist(train_df["review_length"], bins=50, color="steelblue", edgecolor="black")
axes[1].set_title("Review Length Distribution (words)")
axes[1].set_xlabel("Number of Words")
axes[1].set_ylabel("Count")
axes[1].axvline(train_df["review_length"].mean(), color="red", linestyle="--", label=f"Mean: {train_df['review_length'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Average review length: {train_df['review_length'].mean():.0f} words")

## **4** - ***Text Preprocessing***

In [ ]:
def clean_text(text):
    """
    Basic text cleaning for the traditional ML models.
    BERT handles its own tokenization so this is only used for TF-IDF models.
    """
    # Remove HTML tags (IMDB reviews sometimes have <br /> tags)
    text = re.sub(r"<.*?>", " ", text)
    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text


# We use a sample to keep training fast
# Using the full 25k takes much longer
print(f"Using {SAMPLE_SIZE} samples for training (change SAMPLE_SIZE in settings for more)")

sample_train = train_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
sample_test  = test_df.sample(n=1000, random_state=RANDOM_STATE).reset_index(drop=True)

# Clean the text
print("Cleaning text...")
sample_train["clean_text"] = sample_train["text"].apply(clean_text)
sample_test["clean_text"]  = sample_test["text"].apply(clean_text)

print("\nBefore cleaning:")
print(sample_train["text"].iloc[0][:150])
print("\nAfter cleaning:")
print(sample_train["clean_text"].iloc[0][:150])

## **5** - ***Traditional ML Models***

We use TF-IDF to convert text into numbers and then train three classic models on top of it.

In [ ]:
# TF-IDF Vectorization
# Converts text into a matrix of word importance scores
print("Fitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),   # Use single words and bigrams
    stop_words="english"
)

X_train = vectorizer.fit_transform(sample_train["clean_text"])
X_test  = vectorizer.transform(sample_test["clean_text"])
y_train = sample_train["label"]
y_test  = sample_test["label"]

print(f"Feature matrix shape: {X_train.shape}")
print(f"Each review is now represented as a vector of {X_train.shape[1]} features")

In [ ]:
# Train and evaluate all three traditional models:
ml_results = {}

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Naive Bayes":         MultinomialNB(),
    "SVM":                 LinearSVC(random_state=RANDOM_STATE, max_iter=1000)
}

for name, clf in models.items():
    print(f"Training {name}...")
    start = time.time()

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)

    elapsed = round(time.time() - start, 2)
    ml_results[name] = {"accuracy": acc, "time": elapsed, "model": clf, "preds": preds}

    print(f"  Accuracy: {acc:.4f} | Time: {elapsed}s\n")

print("\nAll traditional models trained!")

In [ ]:
# Show detailed report for the best traditional model:
best_ml_name = max(ml_results, key=lambda k: ml_results[k]["accuracy"])
best_ml = ml_results[best_ml_name]

print(f"Best traditional model: {best_ml_name} ({best_ml['accuracy']:.4f})")
print("\nClassification Report:")
print(classification_report(y_test, best_ml["preds"], target_names=["Negative", "Positive"]))

# Confusion matrix
cm = confusion_matrix(y_test, best_ml["preds"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.title(f"Confusion Matrix — {best_ml_name}")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

## **6** - ***BERT Model***

We use DistilBERT — a smaller, faster version of BERT that still performs very well. Instead of training from scratch we use a version already fine-tuned on sentiment analysis.

In [ ]:
print("Loading DistilBERT sentiment model...")
print("This will download the model weights on first run (~250MB)")

# Using a distilbert model already fine-tuned on SST-2 sentiment data.
# This is standard practice. Fine-tuning BERT from scratch needs much more computing time.
bert_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=512
)

print("BERT model loaded!")

In [ ]:
# Run BERT predictions on the test set.
# We use a smaller subset because BERT is slower than traditional models.
BERT_EVAL_SIZE = 500
bert_test = sample_test.head(BERT_EVAL_SIZE)

print(f"Running BERT on {BERT_EVAL_SIZE} test samples...")
print("This may take a few minutes...")

start = time.time()

bert_preds_raw = bert_pipeline(bert_test["text"].tolist(), batch_size=16)

# Convert BERT output (POSITIVE/NEGATIVE) to 1/0:
bert_preds = [1 if r["label"] == "POSITIVE" else 0 for r in bert_preds_raw]
bert_time = round(time.time() - start, 2)

bert_acc = accuracy_score(bert_test["label"], bert_preds)

print(f"\nBERT Accuracy: {bert_acc:.4f}")
print(f"Time taken: {bert_time}s for {BERT_EVAL_SIZE} samples")

print("\nClassification Report:")
print(classification_report(bert_test["label"], bert_preds, target_names=["Negative", "Positive"]))

## **7** - ***Model Comparison***

In [ ]:
# Building a comparison table:
comparison = []
for name, res in ml_results.items():
    comparison.append({
        "Model": name,
        "Accuracy": round(res["accuracy"], 4),
        "Training Time (s)": res["time"],
        "Type": "Traditional ML"
    })

comparison.append({
    "Model": "DistilBERT",
    "Accuracy": round(bert_acc, 4),
    "Training Time (s)": f"{bert_time} (inference only)",
    "Type": "Transformer"
})

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

# Plot accuracy comparison:
plt.figure(figsize=(9, 5))
colors = ["steelblue", "steelblue", "steelblue", "#e67e22"]
bars = plt.bar(comparison_df["Model"], comparison_df["Accuracy"], color=colors, edgecolor="black")

# Adding value labels on top of bars:
for bar, val in zip(bars, comparison_df["Accuracy"]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f"{val:.4f}", ha="center", va="bottom", fontsize=10)

plt.title("Model Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0.7, 1.0)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## **8** - ***Gradio Dashboard***

A simple interactive demo where you can type any text and get a sentiment prediction from all models.

In [ ]:
import gradio as gr

def predict_sentiment(text):
    """
    Runs the input text through all models and returns their predictions.
    """
    if not text.strip():
        return "Please enter some text!"

    clean = clean_text(text)
    tfidf_vec = vectorizer.transform([clean])

    results = []

    # Traditional ML predictions:
    for name, res in ml_results.items():
        pred = res["model"].predict(tfidf_vec)[0]
        label = "Positive" if pred == 1 else "Negative"
        results.append(f"{name}: {label}")

    # BERT prediction:
    bert_result = bert_pipeline(text[:512])[0]
    bert_label = bert_result["label"].capitalize()
    bert_conf = round(bert_result["score"] * 100, 1)
    results.append(f"DistilBERT: {bert_label} ({bert_conf}% confidence)")

    return "\n".join(results)


# Building the Gradio interface:
demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type a movie review or any text here...",
        label="Input Text"
    ),
    outputs=gr.Textbox(label="Sentiment Predictions"),
    title="Sentiment Analysis — ML vs BERT",
    description="Compare predictions from Logistic Regression, Naive Bayes, SVM and DistilBERT",
    examples=[
        ["This movie was absolutely fantastic! The acting was superb and the story kept me engaged throughout."],
        ["Terrible film. Waste of time and money. The plot made no sense at all."],
        ["It was okay, nothing special. Some good moments but overall pretty average."]
    ]
)

demo.launch(share=True)  # share=True gives a public link that works outside Colab

## ****Things You Can Change!***

- `SAMPLE_SIZE` — increase to 10000 or 25000 for better model accuracy (slower training)
- `MAX_FEATURES` — more features for TF-IDF means more detail but slower
- `BERT_EVAL_SIZE` — increase to evaluate BERT on more samples
- `ngram_range=(1, 2)` in TF-IDF — try `(1, 3)` to include trigrams
- The BERT model — swap `distilbert-base-uncased-finetuned-sst-2-english` for other sentiment models on HuggingFace
